In [7]:
import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
df['review'][3].lower()

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.<br /><br />ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

In [ ]:
# text preprocessing: convert all reviews to lowercase
df['review'] = df['review'].str.lower()

In [12]:
# using regex to remove HTML tags
import re
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)
df['review'] = df['review'].apply(remove_html_tags)

df['review'][3]

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

In [13]:
# removing urls from the reviews
def remove_urls(text):
    url_pattern = re.compile(r'http\S+|www\S+')
    return url_pattern.sub(r'', text)

df['review'] = df['review'].apply(remove_urls)

In [ ]:
# remove punctuation
import string, time
string.punctuation

exclude = string.punctuation + '“”‘’—…'
def remove_punctuation(text):
    for char in exclude:
        text = text.replace(char, '')
    return text

# def remove_punctuation(text):
#     return text.translate(str.maketrans('', '', exclude)) ## this is faster method but not working for some special punctuation like “ ” ‘ ’ — …

df['review'] = df['review'].apply(remove_punctuation)
df['review'][3]

'basically theres a family where a little boy jake thinks theres a zombie in his closet  his parents are fighting all the timethis movie is slower than a soap opera and suddenly jake decides to become rambo and kill the zombieok first of all when youre going to make a film you must decide if its a thriller or a drama as a drama the movie is watchable parents are divorcing  arguing like in real life and then we have jake with his closet which totally ruins all the film i expected to see a boogeyman similar movie and instead i watched a drama with some meaningless thriller spots3 out of 10 just for the well playing parents  descent dialogs as for the shots with jake just ignore them'

In [15]:
# chat word treatment
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired",
}

In [16]:
def chat_conversion(text):
    words = text.split()
    new_text = []
    for word in words:
        if word.upper() in chat_words:
            new_text.append(chat_words[word.upper()])
        else:
            new_text.append(word)
    return ' '.join(new_text)

In [17]:
chat_conversion("btw im going to the mall asap")

'By The Way im going to the mall As Soon As Possible'

In [25]:
# text spell correction
from textblob import TextBlob
incorrecct_text = "This moviw ws realy awesome. I reaally lov iet and wuld recommend it to evryone."
tectblb = TextBlob(incorrecct_text)
corrected_text = tectblb.correct().string
print("Incorrect Text: ", incorrecct_text)
print("Corrected Text: ", corrected_text)

Incorrect Text:  This moviw ws realy awesome. I reaally lov iet and wuld recommend it to evryone.
Corrected Text:  His movie was really awesome. I really love it and would recommend it to everyone.


In [35]:
# removing stopwords
import nltk
stop_words = set(nltk.corpus.stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

sentence = "This is a sample sentence showing off the stop words filtration."
print("Original Sentence: ", sentence)
print("After Stopword Removal: ", remove_stopwords(sentence))

df['review'] = df['review'].apply(remove_stopwords)
df['review'][3]

Original Sentence:  This is a sample sentence showing off the stop words filtration.
After Stopword Removal:  This sample sentence showing stop words filtration.


'basically theres family little boy jake thinks theres zombie closet parents fighting timethis movie slower soap opera suddenly jake decides become rambo kill zombieok first youre going make film must decide thriller drama drama movie watchable parents divorcing arguing like real life jake closet totally ruins film expected see boogeyman similar movie instead watched drama meaningless thriller spots3 10 well playing parents descent dialogs shots jake ignore'

In [38]:
# handling emoticons and emojis
import emoji
def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')
    
trysen = "im very sad 😢 today!"
ans = remove_emojis(trysen)
print("Original Text: ", trysen)
print("After Removing Emojis: ", ans)

print(emoji.demojize(trysen))
# df['review'] = df['review'].apply(remove_emojis)
# df['review'][3]

Original Text:  im very sad 😢 today!
After Removing Emojis:  im very sad  today!
im very sad :crying_face: today!


In [ ]:
# tokenization can be done using split method too but it fails in higher level, regex can be used but they are too complicate
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
def tokenize_text(text):
    return word_tokenize(text)

print(tokenize_text("im in new delhi"))

['im', 'in', 'new', 'delhi']


[nltk_data] Downloading package punkt_tab to /home/mohitm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [49]:
# tokenisation using spacy
import spacy
nlp = spacy.load('en_core_web_sm')
sen1 = "i have to 5km far"
doc1 = nlp(sen1)

for token in doc1:
    print(token)

i
have
to
5
km
far


In [50]:
# stemming -> chopping the postfixes
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
def stem_words(text):
    return " ".join([ps.stem(word) for word in text.split()])

sample = "walk walks change changing changed"
stem_words(sample)

'walk walk chang chang chang'

In [57]:
# lemmatization -> chopping and returning root form which is dictionary valid
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemma = WordNetLemmatizer()

def lemmatize(text):
    return " ".join([lemma.lemmatize(word,pos='v') for word in text.split()]) # we have to specify a part of speech using pos = 'v' which is verb

sample = "walk walks change changing changed"
lemmatize(sample)

[nltk_data] Downloading package wordnet to /home/mohitm/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


'walk walk change change change'